# 🚀 ACE-Net Master Baseline Model Training & Evaluation
### End-to-End Multimodal Deepfake Consistency Training on 14k Preprocessed Dataset

### 🌟 Streamlined 5-Step Workflow:
1. **Step 1:** Connect to T4 GPU & Mount Google Drive
2. **Step 2:** Clone or Pull Repository & Install Dependencies
3. **Step 3:** Fast Unzip Master Dataset (`baseline_features_all.zip`) to NVMe Local SSD (~20s)
4. **Step 4:** Pre-Flight Local Dataset Integrity Audit (~1s)
5. **Step 5:** Launch Stage-2 Baseline Training Engine with Live Progress Bar & Checkpointing


## Step 1: Connect to T4 GPU & Mount Google Drive

In [ ]:
from google.colab import drive
import torch

drive.mount('/content/drive')
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name  :', torch.cuda.get_device_name(0))
else:
    print('⚠️ WARNING: GPU is not enabled! Go to Runtime > Change runtime type > T4 GPU!')


## Step 2: Clone or Pull Repository & Install Core Dependencies

In [ ]:
import os
%cd /content
if not os.path.exists('/content/Baseline_Training'):
    !git clone https://github.com/gjvlio/Baseline_Training.git
%cd /content/Baseline_Training
!git checkout feat/baseline-preprocessing-jc
!git pull
!pip install -q scikit-learn transformers
print('✅ Repository and dependencies ready!')


## Step 3: Fast Unzip Master Dataset to Colab Local NVMe SSD (~20 Seconds)
Directly extracts `baseline_features_all.zip` (TRAIN + VAL + TEST) from Google Drive to local SSD.

In [ ]:
import os, time
from pathlib import Path

DRIVE_ZIP = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training/baseline_features_all.zip')
LOCAL_ROOT = Path('/content/preprocessed_local')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

# Skip extraction if already unzipped on this VM
if (LOCAL_ROOT / 'TRAIN').exists() and (LOCAL_ROOT / 'VAL').exists() and (LOCAL_ROOT / 'TEST').exists():
    print('⚡ Dataset already present on Local NVMe SSD! Skipping unzip.')
else:
    if not DRIVE_ZIP.exists():
        raise FileNotFoundError(f'❌ Hindi mahanap ang zip archive sa Drive: {DRIVE_ZIP}')
    print('=' * 75)
    zip_gb = DRIVE_ZIP.stat().st_size / (1024**3)
    print(f'📦 Extracting Master Archive: {DRIVE_ZIP} ({zip_gb:.2f} GB)...')
    t0 = time.time()
    !unzip -q -o "{DRIVE_ZIP}" -d "{LOCAL_ROOT}"
    print(f'✅ Extracted to Local NVMe SSD in {time.time() - t0:.1f}s!')
    print('=' * 75)


## Step 4: [PRE-FLIGHT AUDIT] Fast Dataset Integrity Check (~1 Second)
Verifies 100% presence of Audio, Text, and Visual keyframes across TRAIN, VAL, and TEST on NVMe.

In [ ]:
import csv, time
from pathlib import Path

LOCAL_ROOT = Path('/content/preprocessed_local')
REPO_ROOT = Path('/content/Baseline_Training')
MANIFEST_DIR = REPO_ROOT / 'Manifests' / 'final_manifest_jc'

manifest_roots = [
    REPO_ROOT / 'data' / 'manifests' / 'shards',
    REPO_ROOT / 'Manifests' / 'shards',
    REPO_ROOT / 'data' / 'manifests' / 'eval_shards',
    REPO_ROOT / 'Manifests' / 'eval_shards',
]

clip_to_rel = {}
for mroot in manifest_roots:
    if mroot.exists():
        for mf in mroot.glob('**/*_manifest.csv'):
            shard_name = mf.stem.replace('_manifest', '')
            posix_p = mf.as_posix().lower()
            if '/test/' in posix_p:
                split = 'TEST'
            elif '/val/' in posix_p:
                split = 'VAL'
            else:
                split = 'TRAIN'
                
            if split == 'TRAIN':
                group = mf.parent.name
                rel_shard = Path('TRAIN') / group / 'shards' / shard_name
            else:
                rel_shard = Path(split) / shard_name / 'shards' / 'shard_0001'
                
            try:
                with open(mf, newline='', encoding='utf-8') as f:
                    for r in csv.DictReader(f):
                        c = r.get('clip_id')
                        if c:
                            clip_to_rel[c] = (split, rel_shard)
            except Exception:
                pass

splits = [
    ('TRAIN', MANIFEST_DIR / 'final_train_manifest.csv'),
    ('VAL',   MANIFEST_DIR / 'final_val_manifest.csv'),
    ('TEST',  MANIFEST_DIR / 'final_test_manifest.csv')
]

print('=' * 85)
print('🔍 PRE-FLIGHT LOCAL DATASET INTEGRITY AUDIT (TRAIN + VAL + TEST)')
print('=' * 85)
print(f"{'Split':<7} | {'Total Clips':<11} | {'Audio %':<9} | {'Text %':<9} | {'Visual %':<10} | {'Status'}")
print('-' * 85)

t0 = time.time()
all_ready = True
for split_name, mf_path in splits:
    total_clips, audio_found, text_found, visual_found = 0, 0, 0, 0
    with open(mf_path, newline='', encoding='utf-8') as f:
        for r in csv.DictReader(f):
            total_clips += 1
            cid = r.get('clip_id')
            info = clip_to_rel.get(cid)
            if not info:
                continue
            _, rel_shard = info
            sdir = LOCAL_ROOT / rel_shard
            if (sdir / 'audio' / f'{cid}_melspec.npy').exists():
                audio_found += 1
            if (sdir / 'text' / f'{cid}_input_ids.npy').exists():
                text_found += 1
            if (sdir / 'visual' / cid / 'frame_00000.jpg').exists() or (sdir / 'visual' / cid).exists():
                visual_found += 1
    a_pct = (audio_found / max(total_clips, 1)) * 100
    t_pct = (text_found / max(total_clips, 1)) * 100
    v_pct = (visual_found / max(total_clips, 1)) * 100
    is_ok = (a_pct >= 99.9 and t_pct >= 99.9 and v_pct >= 99.9)
    if not is_ok:
        all_ready = False
    status = '✅ 100% Complete' if is_ok else f'⚠️ Missing ({audio_found}/{total_clips})'
    print(f"{split_name:<7} | {total_clips:<11,} | {a_pct:>7.1f}% | {t_pct:>7.1f}% | {v_pct:>8.1f}% | {status}")

print('=' * 85)
print(f'⏱️ Audit finished in {time.time() - t0:.2f} seconds!')
if all_ready:
    print('🎉 100% HEALTHY! Ready to launch Step 5 Training Engine.')
else:
    print('⚠️ May mga kulang pang files. Suriin ang output sa itaas.')


## Step 5: Run Stage-2 Baseline Training & Evaluation (Live Progress Bar)
Trains for 20 epochs (~0.25s/it, ~2 mins/epoch), evaluates per epoch, auto-saves `best_baseline_model.pth` to Google Drive, and computes official final Test metrics!

In [ ]:
%cd /content/Baseline_Training

# Use -u (unbuffered) so tqdm live progress bar streams in real-time in Colab
!python -u -m src.train_baseline_engine \
    --train-manifest '/content/Baseline_Training/Manifests/final_manifest_jc/final_train_manifest.csv' \
    --val-manifest '/content/Baseline_Training/Manifests/final_manifest_jc/final_val_manifest.csv' \
    --test-manifest '/content/Baseline_Training/Manifests/final_manifest_jc/final_test_manifest.csv' \
    --preprocessed-root '/content/preprocessed_local' \
    --ckpt '/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training/checkpoints/stage2_acenet.pt' \
    --output-dir '/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training/checkpoints' \
    --batch-size 32 \
    --epochs 20 \
    --lr 1e-4 \
    --num-workers 0 \
    --freeze-backbones \
    --device cuda
